In [1]:
import sys
import json
import torch
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from sklearn.model_selection import KFold
from pathlib import Path
from tqdm import tqdm

sys.path.insert(0, str(Path().resolve().parent))
from denoiser.unet_model import UNetDenoiser
from preprocessing.condition_encoder import ConditionEncoder
from diffusion.diffusion import Diffusion
from training import train_step, FinancialDataset
from config import training_config, denoiser_config, diffusion_config, preprocess_config, project_config

In [4]:
# Set seed 
torch.manual_seed(project_config.SEED)
np.random.seed(project_config.SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(project_config.SEED)

# Device
device = project_config.DEVICE

# Hyperparameters
BS = training_config.BATCH_SIZE
LR = training_config.LEARNING_RATE
EP = training_config.EPOCHS
WD = training_config.WEIGHT_DECAY
PU = training_config.P_UNCOND
ES = training_config.EARLY_STOPPING

print(f"Device: {device}")
print(f"Hyperparameters:")
print(f"  Batch size: {BS}")
print(f"  Learning rate: {LR}")
print(f"  Max epochs: {EP}")
print(f"  Weight decay: {WD}")
print(f"  P(uncond): {PU}")
print(f"  Early stopping: {ES}")


Device: cpu
Hyperparameters:
  Batch size: 32
  Learning rate: 0.0001
  Max epochs: 3000
  Weight decay: 0.01
  P(uncond): 0.1
  Early stopping: 100


# Section B: Data Loading


In [5]:
# Load dataset
dataset = FinancialDataset('../data/train/train_data_2d.json')

print(f"\nDataset info:")
print(f"  Total samples: {len(dataset)}")
print(f"  Number of assets: {dataset.num_assets}")
print(f"  Spatial dimensions: {dataset.H}x{dataset.W}")

# Inspect a sample
sample = dataset[0]
print(f"\nSample structure:")
for key, val in sample.items():
    print(f"  {key}: shape {val.shape}, dtype {val.dtype}")


AttributeError: 'list' object has no attribute 'shape'

In [ ]:
# Setup 5-fold cross validation
kfold = KFold(n_splits=5, shuffle=True, random_state=preprocess_config.SEED)

print(f"5-Fold Cross Validation Setup:")
print(f"  Total samples: {len(dataset)}")
print(f"  Folds: 5")
print(f"  Approximate train size per fold: {int(len(dataset) * 0.8)}")
print(f"  Approximate val size per fold: {int(len(dataset) * 0.2)}")

# Section C: Model Initialization Function


In [ ]:
def create_models(device):
    """
    Initialize fresh models for each fold.
    
    Returns:
        model, cond_encoder, diffusion
    """
    # Create U-Net model
    model = UNetDenoiser(
        in_channels=dataset.num_assets,  # 3 assets
        base_channels=denoiser_config.BASE_CHANNELS,
        channel_mult=denoiser_config.CHANNEL_MULT,
        num_res_blocks=denoiser_config.NUM_RES_BLOCKS,
        time_embed_dim=denoiser_config.TIME_EMBED_DIM,
        cond_context_dim=denoiser_config.COND_CONTEXT_DIM,
        num_heads=denoiser_config.CROSS_ATTN_NUM_HEADS,
        dropout=denoiser_config.RES_BLOCK_DROPOUT
    ).to(device)
    
    # Create condition encoder
    cond_encoder = ConditionEncoder(
        num_condition_scalars=preprocess_config.NUM_CONDITION_SCALARS,
        cond_fc_hidden_dim=preprocess_config.COND_FC_HIDDEN_DIM,
        cond_1d_channels=preprocess_config.COND_1D_CHANNELS,
        cond_2d_channels=preprocess_config.COND_2D_CHANNELS,
        cond_output_dim=preprocess_config.COND_OUTPUT_DIM,
        target_shape=preprocess_config.TARGET_SHAPE
    ).to(device)
    
    # Create diffusion process
    diffusion = Diffusion(
        timesteps=diffusion_config.TIMESTEPS,
        beta_schedule=diffusion_config.BETA_SCHEDULE,
        beta_start=diffusion_config.BETA_START,
        beta_end=diffusion_config.BETA_END,
        device=device
    )
    
    # Count parameters
    num_params_model = sum(p.numel() for p in model.parameters() if p.requires_grad)
    num_params_cond = sum(p.numel() for p in cond_encoder.parameters() if p.requires_grad)
    total_params = num_params_model + num_params_cond
    
    print(f"Model parameters: {num_params_model:,}")
    print(f"Condition encoder parameters: {num_params_cond:,}")
    print(f"Total trainable parameters: {total_params:,}")
    
    return model, cond_encoder, diffusion

# Test model creation
print("Testing model creation:")
test_model, test_cond_encoder, test_diffusion = create_models(device)
del test_model, test_cond_encoder, test_diffusion  # Clean up
torch.cuda.empty_cache() if torch.cuda.is_available() else None
print("\nModel creation successful!")

# Section D: Training Loop for Each Fold


In [ ]:
def validate(model, cond_encoder, diffusion, val_loader, device):
    """
    Compute validation loss.
    """
    model.eval()
    cond_encoder.eval()
    
    total_loss = 0.0
    num_batches = 0
    
    with torch.no_grad():
        for batch in val_loader:
            # Get data
            x0 = batch['returns'].to(device)  # (B, C, H, W)
            B = x0.size(0)
            
            # Encode conditions
            cond_tokens = cond_encoder(
                trend=batch['trend'].to(device),
                realized_vol=batch['realized_vol'].to(device),
                interest_rate=batch['interest_rate'].to(device),
                volatility_index=batch['volatility_index'].to(device)
            )
            
            # Sample random timesteps
            t = torch.randint(0, diffusion.T, (B,), device=device)
            
            # Compute loss
            loss = diffusion.loss(model, x0, t, cond_tokens)
            total_loss += loss.item()
            num_batches += 1
    
    model.train()
    cond_encoder.train()
    
    return total_loss / num_batches if num_batches > 0 else 0.0

print("Validation function defined!")

In [ ]:
# Create checkpoints directory
import os
checkpoint_dir = Path('../checkpoints')
checkpoint_dir.mkdir(exist_ok=True, parents=True)
print(f"Checkpoint directory: {checkpoint_dir.absolute()}")

# Storage for results across all folds
all_fold_results = []

In [ ]:
# Main training loop across all folds
for fold_idx, (train_indices, val_indices) in enumerate(kfold.split(dataset)):
    print(f"\n{'='*80}")
    print(f"FOLD {fold_idx + 1}/5")
    print(f"{'='*80}")
    print(f"Train samples: {len(train_indices)}, Val samples: {len(val_indices)}")
    
    # Create data subsets
    train_subset = Subset(dataset, train_indices)
    val_subset = Subset(dataset, val_indices)
    
    # Create data loaders
    train_loader = DataLoader(
        train_subset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0,  # Use 0 for notebook, can increase for scripts
        pin_memory=True if torch.cuda.is_available() else False
    )
    
    val_loader = DataLoader(
        val_subset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        pin_memory=True if torch.cuda.is_available() else False
    )
    
    print(f"Batches per epoch - Train: {len(train_loader)}, Val: {len(val_loader)}")
    
    # Initialize models for this fold
    print("\nInitializing models...")
    model, cond_encoder, diffusion = create_models(device)
    
    # Create optimizer
    optimizer = torch.optim.Adam(
        list(model.parameters()) + list(cond_encoder.parameters()),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY
    )
    
    # Learning rate scheduler
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=EPOCHS
    )
    
    # Training tracking
    train_losses = []
    val_losses = []
    best_val_loss = float('inf')
    epochs_without_improvement = 0
    
    print(f"\nStarting training for fold {fold_idx + 1}...")
    
    # Training epochs
    for epoch in range(EPOCHS):
        # Training phase
        model.train()
        cond_encoder.train()
        
        epoch_train_loss = 0.0
        num_train_batches = 0
        
        pbar = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{EPOCHS}", leave=False)
        for batch in pbar:
            # Get data
            x0 = batch['returns'].to(device)  # (B, C, H, W)
            
            # Prepare conditions
            conditions = {
                'trend': batch['trend'].to(device),
                'realized_vol': batch['realized_vol'].to(device),
                'interest_rate': batch['interest_rate'].to(device),
                'volatility_index': batch['volatility_index'].to(device)
            }
            
            # Training step
            loss = train_step(
                denoiser=model,
                diffusion=diffusion,
                x=x0,
                conditions=conditions,
                cond_encoder=cond_encoder,
                optimizer=optimizer,
                p_uncond=P_UNCOND
            )
            
            epoch_train_loss += loss
            num_train_batches += 1
            
            pbar.set_postfix({"loss": f"{loss:.6f}"})
        
        avg_train_loss = epoch_train_loss / num_train_batches
        train_losses.append(avg_train_loss)
        
        # Validation phase
        avg_val_loss = validate(model, cond_encoder, diffusion, val_loader, device)
        val_losses.append(avg_val_loss)
        
        # Update learning rate
        scheduler.step()
        
        # Print progress
        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"Epoch {epoch + 1}/{EPOCHS} - Train Loss: {avg_train_loss:.6f}, Val Loss: {avg_val_loss:.6f}, LR: {scheduler.get_last_lr()[0]:.6f}")
        
        # Early stopping check
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            epochs_without_improvement = 0
            
            # Save best model for this fold
            checkpoint = {
                'fold': fold_idx,
                'epoch': epoch + 1,
                'model_state_dict': model.state_dict(),
                'cond_encoder_state_dict': cond_encoder.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'train_loss': avg_train_loss,
                'val_loss': avg_val_loss,
                'best_val_loss': best_val_loss
            }
            torch.save(checkpoint, checkpoint_dir / f'best_model_fold_{fold_idx + 1}.pt')
        else:
            epochs_without_improvement += 1
        
        # Early stopping
        if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
            print(f"\nEarly stopping triggered at epoch {epoch + 1}")
            print(f"Best validation loss: {best_val_loss:.6f}")
            break
    
    # Store fold results
    fold_results = {
        'fold': fold_idx + 1,
        'train_losses': train_losses,
        'val_losses': val_losses,
        'best_val_loss': best_val_loss,
        'final_epoch': len(train_losses)
    }
    all_fold_results.append(fold_results)
    
    print(f"\nFold {fold_idx + 1} complete!")
    print(f"  Best validation loss: {best_val_loss:.6f}")
    print(f"  Stopped at epoch: {len(train_losses)}")
    
    # Clean up to free memory
    del model, cond_encoder, optimizer, scheduler
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

print(f"\n{'='*80}")
print("ALL FOLDS COMPLETE!")
print(f"{'='*80}")

# Section E: Results Tracking and Visualization


In [ ]:
# Aggregate results across folds
best_val_losses = [fold['best_val_loss'] for fold in all_fold_results]
final_epochs = [fold['final_epoch'] for fold in all_fold_results]

print("\nCross-Validation Results Summary:")
print(f"{'='*60}")
for i, fold in enumerate(all_fold_results):
    print(f"Fold {i+1}:")
    print(f"  Best validation loss: {fold['best_val_loss']:.6f}")
    print(f"  Training stopped at epoch: {fold['final_epoch']}")

print(f"\n{'='*60}")
print(f"Mean validation loss: {np.mean(best_val_losses):.6f} ± {np.std(best_val_losses):.6f}")
print(f"Best fold: Fold {np.argmin(best_val_losses) + 1} (Val loss: {np.min(best_val_losses):.6f})")
print(f"Mean stopping epoch: {np.mean(final_epochs):.1f} ± {np.std(final_epochs):.1f}")
print(f"{'='*60}")

In [ ]:
# Plot training curves for all folds
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, fold in enumerate(all_fold_results):
    ax = axes[i]
    epochs = range(1, len(fold['train_losses']) + 1)
    
    ax.plot(epochs, fold['train_losses'], label='Train Loss', alpha=0.7)
    ax.plot(epochs, fold['val_losses'], label='Val Loss', alpha=0.7)
    ax.axhline(y=fold['best_val_loss'], color='r', linestyle='--', 
               label=f"Best Val: {fold['best_val_loss']:.6f}", alpha=0.5)
    
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_title(f"Fold {i+1} - Training Progress")
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_yscale('log')  # Log scale for better visualization

# Use the last subplot for summary
ax = axes[5]
ax.bar(range(1, 6), best_val_losses, alpha=0.7, color='steelblue')
ax.axhline(y=np.mean(best_val_losses), color='r', linestyle='--', 
           label=f"Mean: {np.mean(best_val_losses):.6f}", alpha=0.7)
ax.set_xlabel('Fold')
ax.set_ylabel('Best Validation Loss')
ax.set_title('Best Validation Loss per Fold')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xticks(range(1, 6))

plt.tight_layout()
plt.savefig(checkpoint_dir / 'training_curves.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\nTraining curves saved to {checkpoint_dir / 'training_curves.png'}")

In [ ]:
# Save results summary to JSON
results_summary = {
    'cross_validation': {
        'n_folds': 5,
        'mean_val_loss': float(np.mean(best_val_losses)),
        'std_val_loss': float(np.std(best_val_losses)),
        'best_fold': int(np.argmin(best_val_losses) + 1),
        'best_fold_val_loss': float(np.min(best_val_losses))
    },
    'folds': []
}

for fold in all_fold_results:
    results_summary['folds'].append({
        'fold': fold['fold'],
        'best_val_loss': float(fold['best_val_loss']),
        'final_epoch': int(fold['final_epoch']),
        'final_train_loss': float(fold['train_losses'][-1]),
        'final_val_loss': float(fold['val_losses'][-1])
    })

# Save to JSON
with open(checkpoint_dir / 'training_results.json', 'w') as f:
    json.dump(results_summary, f, indent=2)

print(f"Results summary saved to {checkpoint_dir / 'training_results.json'}")
print("\nTraining complete! Best model is saved as 'best_model_fold_X.pt'")